###Transform Customer Data
    1. remove records with null customer_id
    2. remove exact duplicates.
    3. remove duplicates records with created timestamp.
    4. Cast the columns into correct data type.
    5. Write transform data into silver schema.

###1. Remove records with null customer id

In [0]:
%sql
select * from ginzobox.bronze.v_customers where customer_id is not null;

###2. remove exact duplicates.

In [0]:
%sql
select * from ginzobox.bronze.v_customers where customer_id is not null order by customer_id

In [0]:
%sql
select distinct * from ginzobox.bronze.v_customers where customer_id is not null order by customer_id

###3. remove duplicates records with created timestamp.

In [0]:
%sql
create or replace temporary view v_customers_distinct as 
select distinct * from ginzobox.bronze.v_customers where customer_id is not null order by customer_id

In [0]:
%sql
with cte_max as (select customer_id, max(created_timestamp) as max_time from v_customers_distinct 
group by customer_id,created_timestamp)
select t.* from v_customers_distinct t 
join cte_max c on t.customer_id = c.customer_id and t.created_timestamp = c.max_time;

###4. cast column into correct datatype

In [0]:
%sql
with cte_max as (select customer_id, max(created_timestamp) as max_time from v_customers_distinct 
group by customer_id,created_timestamp)
select cast(t.created_timestamp as timestamp) as created_timestamp ,
t.customer_id,
t.customer_name,
cast(t.date_of_birth as date) as date_of_birth,
t.email,
cast(t.member_since as date) as member_since,
t.telephone,
t.file_path from v_customers_distinct t 
join cte_max c on t.customer_id = c.customer_id and t.created_timestamp = c.max_time;

###5. Create delta table

In [0]:
%sql
create table ginzobox.silver.customers as 
with cte_max as (select customer_id, max(created_timestamp) as max_time from v_customers_distinct 
group by customer_id,created_timestamp)
select cast(t.created_timestamp as timestamp) as created_timestamp ,
t.customer_id,
t.customer_name,
cast(t.date_of_birth as date) as date_of_birth,
t.email,
cast(t.member_since as date) as member_since,
t.telephone,
t.file_path from v_customers_distinct t 
join cte_max c on t.customer_id = c.customer_id and t.created_timestamp = c.max_time;

In [0]:
%sql
select * from ginzobox.silver.customers;

In [0]:
%sql describe extended ginzobox.silver.customers